# Fama-French 3-factor model

The Fama-French 3-factor model is an extension of the Capital Asset Pricing Model (CAPM) that includes three factors to explain stock returns: market risk, size, and value. The model is expressed as:

$$R_i - R_f = \alpha + \beta_m (R_m - R_f) + \beta_s SMB + \beta_v HML + \varepsilon_i$$

Where:
- $R_i$: Return of the asset
- $R_f$: Risk-free rate
- $R_m$: Return of the market portfolio
- $SMB$ (Small Minus Big): The return difference between small-cap and large-cap stocks
- $HML$ (High Minus Low): The return difference between high book-to-market and low book-to-market stocks
- $\alpha$: Intercept (alpha)
- $\beta_m$: Coefficient for market risk
- $\beta_s$: Coefficient for size factor
- $\beta_v$: Coefficient for value factor
- $\varepsilon_i$: Error term

The Fama-French 3-factor model is widely used in finance to analyze the performance of portfolios and to understand the sources of returns. It helps investors to identify whether a portfolio's returns are due to market risk, size, or value factors, and it can be used to evaluate the performance of active fund managers.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm
import pandas_datareader.data as web

To get information of Market - Risk free, Small Minus Big, High Minus Low, we can use the `pandas_datareader` library to fetch the Fama-French factors from the Kenneth French Data Library.

From here, we will get $R_m - R_f$, $SMB$, and $HML$.

In [23]:
ff_data = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='2010-01-01', end='2023-12-31')[0]
ff_data.index = ff_data.index.to_timestamp(how="end").normalize()  # Convert index to datetime and normalize to end of month
ff_data.head()

C:\Users\Peeyush\AppData\Local\Temp\ipykernel_22596\2971946612.py:1: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_data = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='2010-01-01', end='2023-12-31')[0]
C:\Users\Peeyush\AppData\Local\Temp\ipykernel_22596\2971946612.py:1: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  ff_data = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='2010-01-01', end='2023-12-31')[0]


,Mkt-RF,SMB,HML,RF
Date,,,,
2010-01-31,-3.35,0.43,0.33,0.00
2010-02-28,3.39,1.18,3.18,0.00
2010-03-31,6.30,1.46,2.19,0.01
2010-04-30,1.99,4.84,2.96,0.01
2010-05-31,-7.90,0.13,-2.48,0.01


We are going to check the Berkshire Hathaway (BRK-B) stock returns against the Fama-French 3 factors. We will use the `yfinance` library to fetch the stock data and then perform a regression analysis to see how well the Fama-French factors explain the returns of BRK-B.

In [4]:
stock_data = yf.download('BRK-B', start='2010-01-01', end='2023-12-31', progress=False)['Close']
stock_data.head()

Ticker,BRK-B
Date,
2010-01-04,66.220001
2010-01-05,66.540001
2010-01-06,66.199997
2010-01-07,66.459999
2010-01-08,66.440002


Since the data is computed daily, we will convert into monthly data to match the frequency of the Fama-French factors. We will then calculate the excess returns of BRK-B by subtracting the risk-free rate from the stock returns and perform a regression analysis using the Fama-French factors as independent variables.

Now, this is our $R_i$.

In [9]:
stock_monthly = stock_data.resample('ME').last().pct_change().dropna() * 100 # Convert to percentage returns
stock_monthly.head()

Ticker,BRK-B
Date,
2010-02-28,4.841027
2010-03-31,1.422687
2010-04-30,-5.254087
2010-05-31,-8.376619
2010-06-30,12.955349


In [25]:
df = pd.merge(stock_monthly, ff_data, left_index=True, right_index=True)
df.head()

,BRK-B,Mkt-RF,SMB,HML,RF
Date,,,,,
2010-02-28,4.841027,3.39,1.18,3.18,0.00
2010-03-31,1.422687,6.30,1.46,2.19,0.01
2010-04-30,-5.254087,1.99,4.84,2.96,0.01
2010-05-31,-8.376619,-7.90,0.13,-2.48,0.01
2010-06-30,12.955349,-5.56,-1.79,-4.73,0.01


In [26]:
X = df[['Mkt-RF', 'SMB', 'HML']]
y = df['BRK-B'] - df['RF']
sm.OLS(
    y, sm.add_constant(X)
).fit().summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.562
Model:                            OLS   Adj. R-squared:                  0.554
Method:                 Least Squares   F-statistic:                     69.65
Date:                Fri, 20 Feb 2026   Prob (F-statistic):           4.85e-29
Time:                        15:16:05   Log-Likelihood:                -427.75
No. Observations:                 167   AIC:                             863.5
Df Residuals:                     163   BIC:                             876.0
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1275      0.254      0.502      0.616      -0.374       0.629
Mkt-RF         0.8040      0.059     13.601      0.000       0.687       0.921
SMB           -0.5725      0.105     -5.478      0.000      -0.779      -0.366
HML            0.2991      0.075      4.005      0.000       0.152       0.447
==============================================================================
Omnibus:                       47.449   Durbin-Watson:                   2.184
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              192.259
Skew:                           0.993   Prob(JB):                     1.78e-42
Kurtosis:                       7.867   Cond. No.                         4.85
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

1. The "Genius" is an Illusion ($\alpha = 0.1275, p = 0.616$): A p-value of 0.616 means there is a 61.6% chance this Alpha is just random noise. Mathematically, Buffett has zero pure stock-picking Alpha.
2. The Defensive Shield ($\beta_1 = 0.8040$): He takes on less market risk than average. If the market crashes 10%, his portfolio historically only crashes about 8%.
3. The Elephant Hunter ($\beta_2 = -0.5725, p = 0.000$): That massive negative coefficient proves he strictly avoids small companies. He only buys mega-cap juggernauts (Apple, Coca-Cola, Bank of America).
4. The Bargain Shopper ($\beta_3 = 0.2991, p = 0.000$): He loves cash-producing assets that are cheap relative to their book value. He strictly avoids overhyped, expensive growth companies.
5. R-squared ($0.562$): These three factors only explain 56.2% of the variance in Berkshire's returns. There is still 43.8% of the stock's movement that this linear model cannot explain. (This led Fama and French to eventually create a 5-Factor model, adding "Profitability" and "Investment" factors).
6. Prob(JB) ($1.78e-42$): The Jarque-Bera test checks if the residuals (errors) are normally distributed. A p-value that microscopic strictly rejects the null hypothesis. The errors have extreme "fat tails." This perfectly aligns with the reality that financial data is rarely ever distributed in a perfect, straight-line bell curve.

The Quant Resolution (The Gauss-Markov Theorem): 
We do not throw away the linear Fama-French equation, and we do not try to fit a non-linear distribution (like a Gamma or Poisson distribution) to the model.

Why? Because of the Gauss-Markov Theorem. This theorem proves that even if your data has fat tails and skew, the OLS coefficients (your $\beta$ values) are still unbiased. Your conclusion that Buffett avoids Small-Cap and loves Value is still mathematically true.

### The Danger (and The Fix)

While your $\beta$ coefficients are safe, the fat tails absolutely destroy your Standard Errors.

Because the tails are fatter than OLS expects, the model underestimates the true variance of the errors. This means your p-values might be lying to you. It might say a factor is statistically significant ($p < 0.05$) when it actually isn't.

#### How Quants fix it in Python:

We keep the exact same Fama-French linear equation, but we tell the statsmodels solver to use Robust Standard Errors (specifically, Heteroskedasticity and Autocorrelation Consistent, or "HAC" standard errors, often called Newey-West).

This mathematically widens the confidence intervals to account for the extreme fat tails and skewness, giving you the "true" p-values without changing the core economic equation.

Once you run the HAC-corrected model, your $\beta$ numbers will stay exactly the same, but your `std err` and `P>|z|` columns will shift to reflect the harsh, fat-tailed reality of the stock market.

In [29]:
sm.OLS(y, sm.add_constant(X)).fit(cov_type='HAC', cov_kwds={'maxlags':1}).summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.562
Model:                            OLS   Adj. R-squared:                  0.554
Method:                 Least Squares   F-statistic:                     55.52
Date:                Fri, 20 Feb 2026   Prob (F-statistic):           8.83e-25
Time:                        15:38:48   Log-Likelihood:                -427.75
No. Observations:                 167   AIC:                             863.5
Df Residuals:                     163   BIC:                             876.0
Df Model:                           3                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1275      0.238      0.536      0.592      -0.339       0.594
Mkt-RF         0.8040      0.074     10.836      0.000       0.659       0.949
SMB           -0.5725      0.083     -6.912      0.000      -0.735      -0.410
HML            0.2991      0.081      3.712      0.000       0.141       0.457
==============================================================================
Omnibus:                       47.449   Durbin-Watson:                   2.184
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              192.259
Skew:                           0.993   Prob(JB):                     1.78e-42
Kurtosis:                       7.867   Cond. No.                         4.85
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 1 lags and without small sample correction
"""